# Injury Intelligence - RAG Pipeline

Author: Angela Lekivetz

This notebook builds the OSHA narratives into a FAISS (Facebook AI Similarity Search) to store and organize the embeddings. A retrieval function is then used to retrieve the most similar narratives from the FAISS index, and this information is then passed to Claude to generate a response to plain English text. 

No model training is required as we are using the `all-MiniLM-L6-v2` model.

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import anthropic
import os

## 1. Load the Dataset

The cleaned and lemmatized dataset from `1_eda.ipynb` is loaded. 

Our RAG pipeline will be based on the `Abstract Text` column, as we want to give Claude the full, readable original text of the accident.


In [ ]:
df = pd.read_csv('../data/osha_filtered.csv')
print(df.shape)
df.head()

## 2. Build Embeddings

In this section, we will build embeddings for the dataset using the `all-MiniLM-L6-v2` model. This converts the narratives into vector representations, enabling us to determine the similarity between them.

We then store the embeddings using FAISS, a library for efficient similarity search and clustering.

In [ ]:
EMBED_MODEL = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBED_MODEL)

print('Encoding texts...')
embeddings = embedder.encode(df['Abstract Text'].tolist(), show_progress_bar=True, batch_size=64)
embeddings = embeddings.astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f'Index contains {index.ntotal} vectors')

faiss.write_index(index, '../data/injury_index.faiss')
df.to_csv('../data/osha_corpus.csv', index=False)
print('Saved.')

## 3. Retrieval Function

A custom retrieval function is used to encode a user-given text into a vector representation using the same embedding model. The function then queries the FAISS index to find the top-k most similar documents to the user-given text. 

In [ ]:
def retrieve(query, k=5):
    query = query.lower()

    query_embedding = embedder.encode([query]).astype('float32')

    distances, indices = index.search(query_embedding, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'text': row['Abstract Text'],
            'injury_type': row['Event type'],
            'distance': float(dist)
        })
    
    return results

# Test
query = 'What causes hand injuries?'
for result in retrieve(query):
    print(result)

## 4. Generating Answers with Claude

In order to answer user queries into our data, we generate a prompt for each query based on what is returned from our retrieval process. This information is then passed to Claude to generate a response.

In [ ]:
def rag_query(query):
    # Retreive the top 5 results
    results = retrieve(query)

    context = '\n\n'.join([
        f"Injury Type: {r['injury_type']}\nText: {r['text']}"
        for r in results
    ])

    prompt = f"""
    You are an occupational health analyst. 
    Use only the workplace injury records below to answer the question. 
    If the records don't contain enough information, say you are unsure.
    
    INJURY RECORDS:
    {context}

    QUESTION: 
    {query}"""

    client = anthropic.Anthropic()  #

    response = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response.content[0].text


In [ ]:
print(rag_query('What causes hand injuries?'))

In [ ]:
print(rag_query('What are common causes of fall injuries?'))

In [ ]:
print(rag_query('What industries have the most severe injuries?'))